In [ ]:
#import the necessary libraries

import os
import time
import pandas as pd
import geopandas as gpd

from pathlib import Path
from streetview import search_panoramas
from tqdm import tqdm

from shapely.ops import nearest_points


In [ ]:
# SETTINGS
PARCEL_GDB = Path(r"great_lakes_typology/independent_validation/Typology_2.gdb")

PARCEL_LAYER = "Farm_Parcels_GLR_2008_Main"

ROAD_FILE = Path(r"great_lakes_typology/independent_validation/roads/glr_roads.shp")

OUT_DIR = Path(r"great_lakes_typology/independent_validation/streetview_selection") 

OUT_DIR.mkdir(parents=True, exist_ok=True)

ID_FIELD = "CSBID"

MAX_ROAD_DISTANCE = 100
CHUNK_SIZE = 100_000

In [ ]:
# LOAD DATA
print("Loading parcels...")
parcels = gpd.read_file(
    PARCEL_GDB,
    layer=PARCEL_LAYER
)

print("Loading roads...")
roads = gpd.read_file(ROAD_FILE)

parcels = parcels.to_crs(roads.crs)

In [ ]:
# ONE POLYGON / CENTROID PER PARCEL ID
print("Dissolving parcels by CSBID...")

parcels = (
    parcels
    .dissolve(by=ID_FIELD)
    .reset_index()
)

print("Creating centroids...")

centroids = parcels[
    [ID_FIELD, "geometry"]
].copy()

centroids["geometry"] = centroids.geometry.centroid

print(f"Total parcel centroids: {len(centroids):,}")

In [ ]:
# FIND CENTROIDS WITHIN 100 M OF ROADS
near_parts = []

for start in tqdm(
    range(0, len(centroids), CHUNK_SIZE),
    desc="Searching roads",
    unit="chunk"
):

    chunk = centroids.iloc[
        start:start + CHUNK_SIZE
    ].copy()

    matched = gpd.sjoin_nearest(
        chunk,
        roads[["geometry"]],
        how="inner",
        max_distance=MAX_ROAD_DISTANCE,
        distance_col="road_distance_m"
    )

    near_parts.append(matched)


near = gpd.GeoDataFrame(
    pd.concat(
        near_parts,
        ignore_index=True
    ),
    geometry="geometry",
    crs=centroids.crs
)

# Keep one closest road per parcel
near = (
    near
    .sort_values("road_distance_m")
    .drop_duplicates(subset=ID_FIELD)
    .reset_index(drop=True)
)

print(f"Centroids within {MAX_ROAD_DISTANCE} m of road: {len(near):,}")

In [ ]:
# GET NEAREST POINT ON MATCHED ROAD
print("Calculating nearest road query points...")

near["road_geometry"] = [
    roads.geometry.iloc[idx]
    for idx in near["index_right"]
]

near["road_point"] = [
    nearest_points(point, road)[1]
    for point, road in tqdm(
        zip(
            near.geometry,
            near["road_geometry"]
        ),
        total=len(near),
        desc="Nearest road point",
        unit="parcel"
    )
]

In [ ]:
# CONVERT QUERY POINTS TO LAT/LON
road_points = gpd.GeoDataFrame(
    near[[ID_FIELD]].copy(),
    geometry=near["road_point"],
    crs=roads.crs
).to_crs(4326)

near["query_lon"] = road_points.geometry.x.values
near["query_lat"] = road_points.geometry.y.values

# Parcel centroid coordinates for manual inspection
centroids_ll = near.to_crs(4326)

near["centroid_lon"] = centroids_ll.geometry.x.values
near["centroid_lat"] = centroids_ll.geometry.y.values

In [ ]:
# SAVE PREPARED DATA
prepared = near[
    [
        ID_FIELD,
        "road_distance_m",
        "centroid_lat",
        "centroid_lon",
        "query_lat",
        "query_lon",
        "geometry"
    ]
].copy()

OUT_FILE = OUT_DIR / "streetview_prepared_parcels.parquet"

prepared.to_parquet(
    OUT_FILE,
    index=False
)

print()
print("Preprocessing complete.")
print(f"Prepared parcels: {len(prepared):,}")
print(f"Saved to: {OUT_FILE}")

Find out the coordinates of the points nearest to the road

In [ ]:
# Settings
OUT_DIR = Path("great_lakes_typology/independent_validation/streetview_selection")
PREPARED_FILE = OUT_DIR / "streetview_prepared_parcels.parquet"

ID_FIELD = "CSBID"
SLEEP_TIME = 1

# Load all parcels
near = gpd.read_parquet(PREPARED_FILE).reset_index(drop=True)

print(f"Total parcels: {len(near):,}")

# Check Street View availability
records = []

In [ ]:
for row in tqdm(near.itertuples(index=False), total=len(near), desc="Checking Street View"):

    parcel_id = getattr(row, ID_FIELD)
    available = 0

    try:
        result = search_panoramas(lat=row.query_lat, lon=row.query_lon)
        available = int(bool(result))

    except Exception as e:
        print(f"Error for {parcel_id}: {e}", flush=True)

    records.append({
        ID_FIELD: parcel_id,
        "streetview_available": available
    })

    time.sleep(SLEEP_TIME)

# Merge and export results
results = near.merge(pd.DataFrame(records), on=ID_FIELD, how="left")

columns = [
    ID_FIELD,
    "centroid_lat",
    "centroid_lon",
    "query_lat",
    "query_lon",
    "road_distance_m",
    "streetview_available"
]

results[columns].to_csv(
    OUT_DIR / "streetview_availability.csv",
    index=False
)

In [ ]:
print("Street View availability check complete.")
print(f"Total parcels: {len(results):,}")
print(f"Street View available: {results['streetview_available'].sum():,}")

Calculate the overall accuracy of integrated typology for the parcel data extracted from the google street view

In [ ]:
file = r"independent_validation\streetview_selection\streetview_points_merge_int_typo_cleaned.csv"

df = pd.read_csv(file)

confusion matrix

In [ ]:
ref = df["Typology"]
pred = df["multi_year_int_typo_label"]

# Confusion matrix
cm = pd.crosstab(
    ref, pred,
    rownames=["Reference"],
    colnames=["Mapped"]
)

print("\nConfusion Matrix:")
print(cm)

In [ ]:
# Overall accuracy
correct = (ref == pred).sum()
total = len(df)
oa = correct / total

print("\nCorrect:", correct)
print("Total:", total)
print(f"Overall Accuracy: {oa * 100:.2f}%")

In [ ]:
# Class-specific accuracy
classes = sorted(set(ref.dropna()) | set(pred.dropna()))

for c in classes:
    TP = ((ref == c) & (pred == c)).sum()
    FN = ((ref == c) & (pred != c)).sum()
    FP = ((ref != c) & (pred == c)).sum()

    PA = TP / (TP + FN) if (TP + FN) > 0 else float("nan")
    UA = TP / (TP + FP) if (TP + FP) > 0 else float("nan")

    print(
        f"{c}: n={(ref == c).sum()}, "
        f"PA={PA*100:.2f}%, "
        f"UA={UA*100:.2f}%"
    )